# Clean Dataset

This notebook loads the raw dataset and applies a series of deterministic cleaning steps while reporting row counts after each operation.

In [ ]:
import pandas as pd
import ast
from nltk.corpus import wordnet
import re


def safe_parse_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


csv_path = "../data/dataset_full.csv"
df = pd.read_csv(csv_path, low_memory=False)
df.drop(["Unnamed: 0"], axis=1, inplace=True)

print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2231142


,title,ingredients,directions,link,source,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [2]:
simplify_dict = {"bite size shredded rice biscuits": "biscuits"}


def preprocess_ner(ner_list):
    return [re.sub(r"[^a-zA-Z\s]", "", ner.lower()).strip() for ner in ner_list]


def singularize_word(word):
    singular = word
    # Using WordNet to get singular form if available
    for syn in wordnet.synsets(word):
        if syn.name().split(" ")[0] == word:

            singular = syn.lemmas()[0].name()
            break
    return singular


def simplify_ingredients(ingredient_list):
    simplified = []
    for ing in ingredient_list:
        if not ing.strip():  # Skip empty strings
            continue
        ing = ing.lower()  # Convert to lowercase
        for key, value in simplify_dict.items():
            if key in ing:
                ing = value
        simplified.append(ing)
    return simplified


df["NER"] = df["NER"].apply(eval).apply(preprocess_ner)
df["simplified_NER"] = df["NER"].apply(simplify_ingredients)

print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2231142


,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[brown sugar, milk, vanilla, nuts, butter, bit...","[brown sugar, milk, vanilla, nuts, butter, bis..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[beef, chicken breasts, cream of mushroom soup...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[frozen corn, cream cheese, butter, garlic pow...","[frozen corn, cream cheese, butter, garlic pow..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[chicken, chicken gravy, cream of mushroom sou...","[chicken, chicken gravy, cream of mushroom sou..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[peanut butter, graham cracker crumbs, butter,...","[peanut butter, graham cracker crumbs, butter,..."


## Cleaning Steps

Steps below are applied in order. Each step prints the remaining row count so you can track how much data is removed.

In [3]:
# 1) Deduplicate duplicate rows (handle list columns safely)
tmp = df.copy()
list_cols = [
    col for col in tmp.columns if tmp[col].map(lambda v: isinstance(v, list)).any()
]
for col in list_cols:
    tmp[col] = tmp[col].map(lambda v: tuple(v) if isinstance(v, list) else v)
df = df.loc[~tmp.duplicated()].copy()
print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2231142


,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[brown sugar, milk, vanilla, nuts, butter, bit...","[brown sugar, milk, vanilla, nuts, butter, bis..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[beef, chicken breasts, cream of mushroom soup...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[frozen corn, cream cheese, butter, garlic pow...","[frozen corn, cream cheese, butter, garlic pow..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[chicken, chicken gravy, cream of mushroom sou...","[chicken, chicken gravy, cream of mushroom sou..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[peanut butter, graham cracker crumbs, butter,...","[peanut butter, graham cracker crumbs, butter,..."


In [4]:
# 2) Drop all missing rows
df = df.dropna(how="any")
print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2231141


,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[brown sugar, milk, vanilla, nuts, butter, bit...","[brown sugar, milk, vanilla, nuts, butter, bis..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[beef, chicken breasts, cream of mushroom soup...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[frozen corn, cream cheese, butter, garlic pow...","[frozen corn, cream cheese, butter, garlic pow..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[chicken, chicken gravy, cream of mushroom sou...","[chicken, chicken gravy, cream of mushroom sou..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[peanut butter, graham cracker crumbs, butter,...","[peanut butter, graham cracker crumbs, butter,..."


In [5]:
# 3) Drop all empty strings (including whitespace-only)
empty_mask = df.apply(
    lambda col: col.map(lambda v: isinstance(v, str) and v.strip() == "")
)
df = df.loc[~empty_mask.any(axis=1)].copy()
print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2231141


,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[brown sugar, milk, vanilla, nuts, butter, bit...","[brown sugar, milk, vanilla, nuts, butter, bis..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[beef, chicken breasts, cream of mushroom soup...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[frozen corn, cream cheese, butter, garlic pow...","[frozen corn, cream cheese, butter, garlic pow..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[chicken, chicken gravy, cream of mushroom sou...","[chicken, chicken gravy, cream of mushroom sou..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[peanut butter, graham cracker crumbs, butter,...","[peanut butter, graham cracker crumbs, butter,..."


In [6]:
# 4) Drop all zero-length lists (including ingredients and NER)
def _to_list(v):
    if isinstance(v, list):
        return v
    return safe_parse_list(v)


list_cols = [
    col
    for col in df.columns
    if df[col]
    .dropna()
    .astype(str)
    .head(50)
    .map(lambda s: s.strip().startswith("[") and s.strip().endswith("]"))
    .any()
]
for col in ["ingredients", "NER"]:
    if col in df.columns and col not in list_cols:
        list_cols.append(col)

if list_cols:
    zero_len_mask = df[list_cols].apply(
        lambda col: col.map(lambda v: len(_to_list(v)) == 0)
    )
    df = df.loc[~zero_len_mask.any(axis=1)].copy()

print(f"Number of rows: {df.shape[0]}")
df.head()

Number of rows: 2230540


,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[brown sugar, milk, vanilla, nuts, butter, bit...","[brown sugar, milk, vanilla, nuts, butter, bis..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[beef, chicken breasts, cream of mushroom soup...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[frozen corn, cream cheese, butter, garlic pow...","[frozen corn, cream cheese, butter, garlic pow..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[chicken, chicken gravy, cream of mushroom sou...","[chicken, chicken gravy, cream of mushroom sou..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[peanut butter, graham cracker crumbs, butter,...","[peanut butter, graham cracker crumbs, butter,..."


## Export

After cleaning, save the dataset for downstream use.

In [7]:
df.to_csv("../data/dataset_full_cleaned.csv")